# Task 3.1: Exploratory Data Analysis (EDA) and Preprocessing Pipeline
## AuraCart Retail Analytics - Production-Grade E-Commerce Analytics

This notebook encapsulates all statistical visualizations, feature engineering logic, correlation analyses, and the final serialization of the Scikit-learn preprocessing pipeline objects as required by the Project Specification.

In [ ]:
# Install all required dependencies into the current kernel environment
import sys
!{sys.executable} -m pip install pandas numpy matplotlib seaborn datasets scikit-learn joblib --quiet
print("All dependencies installed successfully.")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import joblib
import os

# Set visual style
sns.set_theme(style="whitegrid")
print("All libraries imported successfully.")

## 1. Data Characterization and Loading
We utilize the 'E-commerce Customer Order Behavior Dataset' from Hugging Face hub (10,000 real e-commerce transactions).

In [ ]:
print("Loading real AuraCart dataset from HuggingFace...")
dataset = load_dataset("millat/e-commerce-orders")
df = pd.DataFrame(dataset['train'])
print(f"Dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"Columns: {list(df.columns)}")
df.head()

In [ ]:
print("=== Dataset Summary Statistics ===")
df.describe()

In [ ]:
print("=== Missing Values Check ===")
print(df.isnull().sum())

## 2. Exploratory Data Analysis (EDA)
### 2.1 Distribution of Continuous Variables
We examine the distribution of 'price' and 'quantity' to detect skewness and outliers.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df['price'], kde=True, color='royalblue', ax=axes[0])
axes[0].set_title('Distribution of Transaction Price', fontsize=14)
axes[0].set_xlabel('Price (USD)')

sns.histplot(df['quantity'], kde=True, color='coral', ax=axes[1])
axes[1].set_title('Distribution of Quantity', fontsize=14)
axes[1].set_xlabel('Quantity')

plt.tight_layout()
plt.savefig('../report_screenshots/1b_price_quantity_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved.")

### 2.2 Feature Correlation Analysis
We generate a correlation matrix to examine relationships between numerical features.

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns
corr = df[numeric_cols].corr()

plt.figure(figsize=(10, 7))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f', mask=mask, linewidths=0.5)
plt.title('Feature Correlation Matrix (AuraCart Dataset)', fontsize=16)
plt.tight_layout()
plt.savefig('../report_screenshots/1_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print("Heatmap saved.")

### 2.3 Categorical Variable Analysis
We visualize the distribution of delivery status and customer segments.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(16, 6))

delivery_counts = df['delivery_status'].value_counts()
ax[0].pie(delivery_counts.values, labels=delivery_counts.index, autopct='%1.1f%%', startangle=90)
ax[0].set_title('Delivery Status Distribution', fontsize=14)

segment_counts = df['customer_segment'].value_counts()
ax[1].pie(segment_counts.values, labels=segment_counts.index, autopct='%1.1f%%', startangle=90)
ax[1].set_title('Customer Segment Distribution', fontsize=14)

plt.tight_layout()
plt.savefig('../report_screenshots/2_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("Class distribution chart saved.")

## 3. Preprocessing Pipeline
We build a reproducible Scikit-learn Pipeline for categorical encoding and feature scaling.

In [ ]:
numeric_features = ['quantity', 'price']
categorical_features = ['category', 'payment_method', 'device_type', 'channel', 'delivery_status', 'customer_segment']

# Filter to only existing columns
numeric_features = [c for c in numeric_features if c in df.columns]
categorical_features = [c for c in categorical_features if c in df.columns]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
    ])

full_pipeline = Pipeline(steps=[('preprocessor', preprocessor)])

# Fit on the actual data
X_raw = df[numeric_features + categorical_features]
full_pipeline.fit(X_raw)
print("Preprocessing pipeline built and fitted successfully.")

# Save the preprocessing artifact
os.makedirs('../artifacts', exist_ok=True)
joblib.dump(full_pipeline, '../artifacts/preprocessor.joblib')
print("Artifact saved to: ../artifacts/preprocessor.joblib")